In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langchain.agents.structured_output import ToolStrategy
from typing import Literal, Dict, Any
from dotenv import load_dotenv
from pydantic import BaseModel, Field, model_validator
load_dotenv()

class ModelResponseFormat(BaseModel):
    """Agent Response format"""
    information: str
    claim: str
    source_quote: str = Field(description="the source information used")


model = ChatGroq(
    model="openai/gpt-oss-20b"
)
model_with_structure = model.with_structured_output(ModelResponseFormat)

@tool
def web_search(query: str, max_results: int = 2, topics: Literal["general", "news", "finance" ] = "general", include_raw_content: bool = True) -> Dict[str, Any]:
    """web search tool that reterives the data from web"""
    response =  TavilySearch(topic=topics, max_results= 1, include_raw_content=False).invoke(query)
    # Format only the essential fields into a compact string
    clean_results = []
    for item in response.get("results", []):
        clean_results.append(
            f"Title: {item.get('title')}\n"
            f"URL: {item.get('url')}\n"
            f"Snippet: {item.get('content')[:500]}"
        )
        
    print("websearch result : ", clean_results)
    return "\n\n---\n\n".join(clean_results)


In [ ]:
model.invoke("what is the remaining token i have")

In [ ]:
"""
Implement Quote Verification
tell the agent to return the claim along with source quote 
compare with reterived document 
if source quote in reterived document 
returns Evidence Found
else return evidence not found
"""

agent = create_agent(
    model=model,
    system_prompt="""
    you are a helpful assistant which fetches the current data using web search tool when asked, 
    only reterive maximum 3 websearch results and relay on it for evidence grounded answer
    and return the response claim along with source claim which you used for generating the respone in provided structure.
    """,
    tools=[web_search],
    response_format=ToolStrategy(ModelResponseFormat)
)
agent

In [ ]:
try:
    agent_res = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "According to current Market standards which Car is better in terms of performace and reliability either Volkswagen virtus or Thar rox 4x4"
        }]
    })
except Exception as e:
    print("Status Code:", getattr(e, "status_code", "Unknown"))
    print("Error Body:", getattr(e, "body", str(e)))


In [ ]:
information, claim, source_quote = agent_res['structured_response']

In [ ]:
research_result = {
    "claim": "LiteLLM supports model routing.",
    "source_quote": "LiteLLM supports routing across multiple LLM providers."
}